# Whisper STT Implementation
## Step 1 - Environment
**Objective:** Install required libraries and set up project structure.

**Expected outcome:** You should have the environment set up with necessary directories.

**Checkpoint:** Verify that OpenAI client initializes and directories are created.

Jupyter does not provide a reliable standard notebook-location variable. Set WHISPER_LAB_DIR to the extracted lab folder when launching from a different directory. Optional WHISPER_AUDIO_FILE selects another recording (relative to the lab folder or absolute). Without it, the provided MP3 is used, or a single supported file in audio/ is selected. Multiple candidates require an explicit selection.


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'])


In [ ]:
from pathlib import Path
import os, json, hashlib, subprocess, copy, re, difflib, time, textwrap
from datetime import datetime, timezone
from dotenv import load_dotenv
from openai import OpenAI, APIConnectionError, APIStatusError
from pydub import AudioSegment
import imageio_ffmpeg
import matplotlib.pyplot as plt

BASE = Path(os.getenv('WHISPER_LAB_DIR', str(Path.cwd()))).expanduser().resolve()
audio_setting = os.getenv('WHISPER_AUDIO_FILE')
if audio_setting:
    AUDIO = Path(audio_setting).expanduser()
    if not AUDIO.is_absolute():
        AUDIO = BASE / AUDIO
else:
    AUDIO = BASE / 'project_nimbus_meeting.mp3'
    if not AUDIO.is_file():
        supported = {'.mp3', '.wav', '.m4a', '.mp4', '.mpeg', '.mpga', '.webm'}
        candidates = sorted(path for path in (BASE / 'audio').glob('*')
                            if path.is_file() and path.suffix.lower() in supported)
        if len(candidates) == 1:
            AUDIO = candidates[0]
        elif len(candidates) > 1:
            raise ValueError('Multiple recordings found in audio/. Set WHISPER_AUDIO_FILE explicitly.')
AUDIO = AUDIO.resolve()
OUT = BASE / 'outputs'
for folder in ['responses', 'transcriptions', 'png', 'audio']:
    (OUT / folder).mkdir(parents=True, exist_ok=True)
load_dotenv(BASE / '.env', override=False)
if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError('Add OPENAI_API_KEY to your local .env or environment; do not paste the key into the notebook.')
client = OpenAI(timeout=120, max_retries=2)
MODEL = 'whisper-1'
LANGUAGE = 'en'  # change this if the speech is in another language
PROMPT = 'Project Nimbus, Nimbus.'  # known name from the filename; add only verified terms
FORCE_REFRESH = False  # True repeats paid requests
print('Environment ready. The key was loaded without being displayed.')
print({'lab_directory': str(BASE), 'input_audio': str(AUDIO), 'output_directory': str(OUT)})


## Step 2 - Audio validation and loading
Use the attached MP3 rather than downloading an unrelated sample. Check its size, decode it to mono PCM at 16 kHz, and display its duration and properties. Sample rate normalization simplifies processing but does not improve accuracy by itself. SHA256 links responses to a specific input.

**Objective:** Get a sample meeting audio file for transcription.

**Expected outcome:** You should have an audio file ready for transcription.

**Checkpoint:** Verify that the audio file exists and can be loaded.

In [ ]:
if not AUDIO.is_file():
    raise FileNotFoundError(f'Audio not found: {AUDIO}. Set WHISPER_LAB_DIR or WHISPER_AUDIO_FILE.')
ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()
wav = OUT / 'audio' / 'source.wav'
subprocess.run([ffmpeg, '-hide_banner', '-loglevel', 'error', '-y', '-i', str(AUDIO),
                '-ac', '1', '-ar', '16000', '-c:a', 'pcm_s16le', str(wav)], check=True)
audio = AudioSegment.from_wav(wav)
duration = len(audio) / 1000
if duration <= 0:
    raise ValueError('The decoded audio is empty.')
print(f'Audio loaded successfully: {duration:.2f} seconds ({duration / 60:.2f} minutes).')
print({'file': AUDIO.name, 'bytes': AUDIO.stat().st_size, 'duration_seconds': duration,
       'channels': audio.channels, 'sample_rate': audio.frame_rate})


### Shared API export and PNG functions
Responses are cached by audio and parameters; rerunning does not repeat successful requests. The log saves the time, model, hash, parameters, and request id without keys. An API error stops dependent steps rather than substituting text. JSON preserves the response structure; TXT and SRT are generated locally from the same segments. These are not separate API calls requesting TXT/SRT formats.


In [ ]:
def save_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')

def transcribe(path, label, prompt=None):
    path = Path(path)
    if path.stat().st_size >= 25_000_000:
        raise ValueError('The excerpt is too large: reduce the chunk duration.')
    params = dict(model=MODEL, language=LANGUAGE, temperature=0,
                  response_format='verbose_json', timestamp_granularities=['segment', 'word'])
    if prompt:
        params['prompt'] = prompt
    fingerprint = hashlib.sha256(path.read_bytes() + json.dumps(params, sort_keys=True).encode()).hexdigest()
    target = OUT / 'responses' / (fingerprint + '.json')
    if target.exists() and not FORCE_REFRESH:
        return json.loads(target.read_text(encoding='utf-8'))['response']
    try:
        with path.open('rb') as f:
            response = client.audio.transcriptions.create(file=f, **params)
    except APIConnectionError:
        save_json(OUT / 'run_status.json', {'status':'failed', 'error':'APIConnectionError', 'step':label})
        raise RuntimeError('The API is unavailable. Check your internet connection and proxy. No real result was obtained.') from None
    except APIStatusError as e:
        save_json(OUT / 'run_status.json', {'status':'failed', 'http_status':e.status_code, 'step':label})
        raise RuntimeError(f'The API returned HTTP {e.status_code}. Check access, balance, and limits.') from None
    result = response.model_dump(mode='json')
    if not result.get('text', '').strip():
        raise RuntimeError('The API returned empty text: check whether the audio contains speech.')
    save_json(target, {'provenance':{'status':'real_api_response', 'model':MODEL,
              'utc':datetime.now(timezone.utc).isoformat(), 'audio_sha256':hashlib.sha256(path.read_bytes()).hexdigest(),
              'parameters':params, 'label':label, 'request_id':getattr(response, '_request_id', None)}, 'response':result})
    return result

def stamp(seconds, srt=False):
    milliseconds = round(max(0, float(seconds)) * 1000)
    hours, rest = divmod(milliseconds, 3600000)
    minutes, rest = divmod(rest, 60000)
    secs, ms = divmod(rest, 1000)
    return f'{hours:02}:{minutes:02}:{secs:02}' + (',' if srt else '.') + f'{ms:03}'

def export(result, name):
    segments = result.get('segments', [])
    save_json(OUT / 'transcriptions' / (name + '.json'), result)
    lines = [f"[{stamp(s['start'])} - {stamp(s['end'])}] {s['text'].strip()}" for s in segments]
    (OUT / 'transcriptions' / (name + '.txt')).write_text('\n'.join(lines) if lines else result['text'], encoding='utf-8')
    valid_segments = [s for s in segments if s['text'].strip() and s['end'] > s['start']]
    if not valid_segments:
        raise ValueError(f'No nonempty subtitle cues for {name}; SRT export was not written.')
    cues = [f"{i}\n{stamp(s['start'], True)} --> {stamp(s['end'], True)}\n{s['text'].strip()}\n"
            for i, s in enumerate(valid_segments, 1)]
    (OUT / 'transcriptions' / (name + '.srt')).write_text('\n'.join(cues), encoding='utf-8')

def png(title, lines, filename):
    # Pagination prevents long results from being clipped.
    wrapped = []
    for line in lines:
        wrapped.extend(textwrap.wrap(str(line), width=100, replace_whitespace=False) or [''])
    pages = [wrapped[i:i+36] for i in range(0, len(wrapped), 36)] or [[]]
    for i, page in enumerate(pages, 1):
        fig = plt.figure(figsize=(14, 10), facecolor='white')
        fig.text(.04, .95, title + f' — {i}/{len(pages)}', fontsize=17, weight='bold', va='top')
        fig.text(.04, .88, '\n'.join(page), fontsize=11, family='DejaVu Sans Mono', va='top', linespacing=1.5)
        fig.savefig(OUT / 'png' / f'{filename}_{i:02}.png', dpi=150)
        plt.close(fig)


## Step 3 - Basic transcription
Take the first 60 seconds, or the entire recording if it is shorter. verbose_json returns text and both timestamp granularities. The same response serves as the unprompted baseline in step 5 to avoid paying for a repeated request.

**Objective:** Transcribe a short audio file to understand the basic API usage.

**Expected outcome:** You should see the transcribed text and basic information.

**Checkpoint:** Verify that transcription works and you can see the output.

In [ ]:
sample = OUT / 'audio' / 'sample.wav'
audio[:min(60000, len(audio))].export(sample, format='wav')
unprompted = transcribe(sample, 'short_unprompted')
export(unprompted, 'sample_unprompted')
print(f'Unprompted baseline: {sample.name}, first {min(60, duration):.2f} seconds.')
print(unprompted['text'])
png('Successful transcription — real Whisper API response', [unprompted['text']], 'successful_transcription')


## Step 4 - Guided transcription
The prompt is a short contextual vocabulary in the audio language, not an instruction to the model. The name Nimbus is known from the attachment; unknown names are not added. Compare exactly the same WAV, language, model, and temperature. The vocabulary may improve a term's spelling, but it may also bias the result.

**Objective:** Use prompts to guide transcription and improve accuracy.

**Expected outcome:** You should see transcription results that may be more accurate due to the prompt context.

**Checkpoint:** Verify that prompted transcription works and results are saved.

In [ ]:
print(f'Prompt vocabulary: {PROMPT}')
print('Nimbus is verified from the supplied filename; no unverified names were added.')
prompted = transcribe(sample, 'short_prompted', PROMPT)
export(prompted, 'sample_prompted')
print(prompted['text'])


## Step 5 - Unguided transcription and comparison
Use the saved unprompted response from step 3. Display the actual differing excerpts. Text similarity is not accuracy; without a reference transcript, neither approach can be declared better and a reliable WER cannot be calculated. Listen to disputed passages and create reference.txt manually if needed.

**Objective:** Transcribe the same audio without prompts for comparison.

**Expected outcome:** You should see both transcriptions and be able to compare them.

**Checkpoint:** Verify that both approaches work and you can compare results.

Optional evaluation: place a manually verified transcript of **only the same first 60 seconds (or the entire shorter sample)** in reference.txt in the lab directory. Never use an API hypothesis as its own reference. The next cell evaluates case-insensitive word error rate using word tokens, ignoring punctuation. WER is the minimum number of word substitutions, deletions, and insertions divided by the reference word count; it may exceed 100%. Both approaches use the same normalization.


In [ ]:
before, after = unprompted['text'].split(), prompted['text'].split()
changes = []
for op, i, j, k, l in difflib.SequenceMatcher(None, before, after, autojunk=False).get_opcodes():
    if op != 'equal':
        changes.append({'operation':op, 'unprompted':' '.join(before[i:j]), 'prompted':' '.join(after[k:l])})
save_json(OUT / 'comparison.json', {'prompt':PROMPT, 'differences':changes, 'accuracy_claim':'No reference transcript; WER not measured'})
summary = ['Baseline: unprompted (no prompt), reused from Step 3.',
           'Comparison: prompted, using the same audio and model parameters.',
           'Changed excerpts:']
summary += [f"{x['operation']}: {x['unprompted'] or '[empty]'} -> {x['prompted'] or '[empty]'}"
            for x in changes] or ['No differences.']
(OUT / 'comparison_summary.txt').write_text('\n'.join(summary) + '\n', encoding='utf-8')
print('\n'.join(summary))
comparison_lines = ['UNPROMPTED', unprompted['text'], '', 'PROMPTED', prompted['text'], '', 'DIFFERENCES']
comparison_lines += [f"{x['operation']}: {x['unprompted']} -> {x['prompted']}" for x in changes] or ['No differences']
png('Prompted vs unprompted — same audio', comparison_lines, 'prompted_unprompted_comparison')

def word_error_rate(reference, hypothesis):
    def tokens(text):
        return re.findall(r"\w+(?:['’]\w+)*", text.casefold())
    ref, hyp = tokens(reference), tokens(hypothesis)
    if not ref:
        raise ValueError('The manual reference must contain at least one word.')
    previous = list(range(len(hyp) + 1))
    for i, word in enumerate(ref, 1):
        current = [i]
        for j, candidate in enumerate(hyp, 1):
            current.append(min(previous[j] + 1, current[j-1] + 1,
                               previous[j-1] + (word != candidate)))
        previous = current
    return previous[-1] / len(ref)

reference_path = BASE / 'reference.txt'
if reference_path.is_file():
    reference = reference_path.read_text(encoding='utf-8')
    evaluation = {'reference_file': reference_path.name, 'scope': 'same short sample',
                  'unprompted_wer': word_error_rate(reference, unprompted['text']),
                  'prompted_wer': word_error_rate(reference, prompted['text'])}
    save_json(OUT / 'evaluation.json', evaluation)
    print(evaluation)
else:
    print('WER skipped: no manually verified reference.txt is available.')


## Step 6 - Chunking
The create_chunks function accepts a configurable chunk_seconds parameter and validates every exported WAV. Use 60 seconds for the demonstration so that a short meeting also produces several chunks. For recordings lasting several hours, you can set 600 seconds, but the function automatically limits PCM WAV files to less than 25 MB. Chunks are sequential and do not overlap; this avoids duplicates, but a word at a boundary may be affected. A production version should detect pauses or implement overlap with separate deduplication.

**Objective:** Split long audio files into chunks for processing.

**Expected outcome:** You should have audio files split into chunks.

**Checkpoint:** Verify that chunks are created correctly.

In [ ]:
def create_chunks(audio, output_dir, chunk_seconds=60):
    if chunk_seconds <= 0:
        raise ValueError('chunk_seconds must be positive.')
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    bytes_per_second = audio.frame_rate * audio.channels * audio.sample_width
    safe_ms = int(24_000_000 / bytes_per_second * 1000)
    chunk_ms = min(int(chunk_seconds * 1000), safe_ms)
    if chunk_ms <= 0 or len(audio) == 0:
        raise ValueError('Audio and chunk duration must be nonempty.')
    chunks = []
    for i, start in enumerate(range(0, len(audio), chunk_ms)):
        end = min(start + chunk_ms, len(audio))
        path = output_dir / f'chunk_{i:03}.wav'
        audio[start:end].export(path, format='wav')
        if not path.is_file() or not (44 < path.stat().st_size < 25_000_000):
            raise ValueError(f'Exported chunk is missing, empty, or oversized: {path.name}')
        chunks.append({'index':i, 'path':str(path), 'offset':start/1000, 'duration':(end-start)/1000})
    if abs(sum(c['duration'] for c in chunks) - len(audio)/1000) >= .002:
        raise ValueError('Chunk durations do not cover the complete source audio.')
    return chunks

CHUNK_SECONDS = 60  # for long meetings, this can be set to 600
chunks = create_chunks(audio, OUT / 'audio', chunk_seconds=CHUNK_SECONDS)
save_json(OUT / 'chunk_manifest.json', chunks)
print('Chunks created and validated:', len(chunks))


## Step 7 - Chunks and timestamps
Save each chunk's original response. Add the **actual chunk start time** to every segment/word timestamp, rather than the duration of the last chunk. Check the range, order, and containment within the source excerpt; an out-of-range value raises an error. Whisper does not identify speakers; timestamps are model estimates, not manual annotations. The final segment and word start times are also checked for sorted order. chunk_review.txt preserves explicit chunk boundaries for review without inserting separators into the transcript itself.

**Objective:** Transcribe each chunk and combine with proper timestamps.

**Expected outcome:** You should have a complete transcription with timestamps from chunked audio.

**Checkpoint:** Verify that chunked transcription works and timestamps are correct.

In [ ]:
combined = {'text':'', 'segments':[], 'words':[], 'duration':duration, 'model':MODEL,
            'provenance':'Combined from real chunk API responses; timestamps shifted to source timeline'}
chunk_lines = []
texts = []
for chunk in chunks:
    result = transcribe(chunk['path'], f"chunk_{chunk['index']:03}", PROMPT)
    export(result, f"chunk_{chunk['index']:03}")
    texts.append(result['text'].strip())
    chunk_lines.append(f"Chunk {chunk['index']} offset={chunk['offset']:.3f}s duration={chunk['duration']:.3f}s")
    chunk_lines.append(result['text'])
    for kind in ['segments', 'words']:
        previous = -1
        for item in result.get(kind, []):
            if not (0 <= item['start'] <= item['end'] <= chunk['duration'] + .1):
                raise ValueError(f'Invalid local timestamps in {kind}; check the API response.')
            if item['start'] < previous:
                raise ValueError('Timestamps are not ordered.')
            previous = item['start']
            shifted = copy.deepcopy(item)
            shifted['start'] = min(duration, item['start'] + chunk['offset'])
            shifted['end'] = min(duration, item['end'] + chunk['offset'])
            shifted['chunk_index'] = chunk['index']
            if kind == 'segments': shifted['id'] = len(combined[kind])
            combined[kind].append(shifted)
for kind in ['segments', 'words']:
    items = combined[kind]
    if any(item['start'] > following['start'] for item, following in zip(items, items[1:])):
        raise ValueError(f'Combined {kind} timestamps are not sorted.')
combined['text'] = ' '.join(texts)
# Keep explicit boundaries in a separate review file without changing transcript content.
(OUT / 'chunk_review.txt').write_text('\n\n'.join(chunk_lines) + '\n', encoding='utf-8')
png('Chunked transcription — real results', chunk_lines, 'chunked_results')
timestamp_lines = [f"[{stamp(s['start'])} - {stamp(s['end'])}] {s['text'].strip()}" for s in combined['segments']]
timestamp_lines += ['WORD TIMESTAMPS'] + [f"{w['word']} [{w['start']:.3f}s - {w['end']:.3f}s]" for w in combined['words']]
png('Timestamp extraction — global source timeline', timestamp_lines, 'timestamp_extraction')
print('Segments:', len(combined['segments']), 'Words:', len(combined['words']))


## Step 8 - JSON TXT SRT and validation
SRT uses HH:MM:SS,mmm, numbering starting at one, and a blank line between subtitles. TXT contains readable time intervals. JSON preserves segment/word data and provenance. The complete recording is exported only after all chunks finish successfully; partial responses remain available for resuming. SRT cues are numbered consecutively after filtering empty or zero-duration segments. An explicit error identifies an export with no valid cues.

**Objective:** Export transcriptions in readable formats with timestamps.

**Expected outcome:** You should have transcriptions exported in multiple formats.

**Checkpoint:** Verify that exports work and files are readable.

In [ ]:
export(combined, 'meeting_chunked')
reloaded = json.loads((OUT / 'transcriptions' / 'meeting_chunked.json').read_text(encoding='utf-8'))
assert reloaded['text'] == combined['text']
assert all(0 <= x['start'] <= x['end'] <= duration for x in reloaded['segments'])
save_json(OUT / 'run_status.json', {'status':'completed', 'real_api_results':True, 'chunks':len(chunks), 'duration_seconds':duration})
print('Export completed:', OUT / 'transcriptions')
